# Exercise 1: Random Forest Ensemble Concept - Fruit Basket Analogy

The figure in the book shows N bootstrap samples each producing a decision tree that outputs a class (Class-A or Class-B). We demonstrate the ensemble concept using the Breast Cancer dataset: train a RandomForestClassifier and show the class with the highest aggregate vote/probability.

In [1]:
# CH.SC.U4CSE24119 Kavin JS

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

## Load and Prepare Data

In [2]:
data = pd.read_csv('../Data/breast_cancer_master.csv')
print('Shape:', data.shape)
print(data.head())

x = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=41)

sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

Shape: (569, 32)
         id  mean radius  mean texture  mean perimeter  mean area  \
0    842302        17.99         10.38          122.80     1001.0   
1    842517        20.57         17.77          132.90     1326.0   
2  84300903        19.69         21.25          130.00     1203.0   
3  84348301        11.42         20.38           77.58      386.1   
4  84358402        20.29         14.34          135.10     1297.0   

   mean smoothness  mean compactness  mean concavity  mean concave points  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   mean symmetry  ...  worst texture  worst perimeter  worst area  \
0         0.2419  ...          17.33

## Train Random Forest with 10 Trees

In [3]:
model = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)
model.fit(x_train, y_train)

# Show how many trees vote for each class for the first few test samples
y_pred = model.predict(x_test)

# Get individual tree predictions
tree_preds = np.array([tree.predict(x_test) for tree in model.estimators_])
print(f'Number of trees: {len(model.estimators_)}')
print(f'Test samples: {x_test.shape[0]}')

Number of trees: 10
Test samples: 171


## Visualize Voting for First 10 Test Samples

In [4]:
print('Individual tree predictions (rows=trees, cols=test samples):')
print('Tree  | Sample predictions (first 10)')
print('-' * 60)
for i, tree_pred in enumerate(tree_preds[:10]):
    print(f'Tree {i+1:2d} | {tree_pred[:10]}')

print(f'\nAggregate vote for first 10 samples:')
for j in range(min(10, len(y_test))):
    votes = tree_preds[:, j]
    b_count = (votes == 'B').sum()
    m_count = (votes == 'M').sum()
    predicted = y_pred[j]
    actual = y_test[j]
    match = 'OK' if predicted == actual else 'MISS'
    print(f'  Sample {j+1:2d}: B={b_count:2d}, M={m_count:2d} -> Predicted: {predicted} (Actual: {actual}) [{match}]')

Individual tree predictions (rows=trees, cols=test samples):
Tree  | Sample predictions (first 10)
------------------------------------------------------------
Tree  1 | [0. 0. 0. 0. 0. 0. 1. 1. 1. 0.]
Tree  2 | [0. 1. 0. 0. 0. 0. 1. 1. 1. 0.]
Tree  3 | [0. 0. 1. 0. 0. 0. 1. 1. 1. 0.]
Tree  4 | [0. 1. 0. 0. 1. 0. 1. 1. 1. 0.]
Tree  5 | [0. 0. 0. 0. 1. 0. 1. 1. 1. 1.]
Tree  6 | [0. 0. 1. 1. 0. 0. 1. 1. 1. 0.]
Tree  7 | [0. 1. 0. 0. 0. 0. 1. 1. 1. 0.]
Tree  8 | [0. 1. 0. 0. 0. 0. 1. 1. 1. 0.]
Tree  9 | [0. 0. 1. 0. 0. 0. 1. 1. 1. 0.]
Tree 10 | [1. 0. 1. 1. 0. 0. 1. 1. 1. 0.]

Aggregate vote for first 10 samples:
  Sample  1: B= 0, M= 0 -> Predicted: B (Actual: B) [OK]
  Sample  2: B= 0, M= 0 -> Predicted: B (Actual: B) [OK]
  Sample  3: B= 0, M= 0 -> Predicted: B (Actual: B) [OK]
  Sample  4: B= 0, M= 0 -> Predicted: B (Actual: B) [OK]
  Sample  5: B= 0, M= 0 -> Predicted: B (Actual: B) [OK]
  Sample  6: B= 0, M= 0 -> Predicted: B (Actual: B) [OK]
  Sample  7: B= 0, M= 0 -> Predicted: M 

## Ensemble Accuracy

In [5]:
Accuracy_score = accuracy_score(y_test, y_pred)
print(f'\nEnsemble Accuracy: {int(Accuracy_score * 100)}%')


Ensemble Accuracy: 99%


## Probability Estimates

The predict_proba method shows the fraction of trees voting for each class. The class with the highest probability (majority vote) is selected as the final prediction.

In [6]:
proba = model.predict_proba(x_test)
print('Probability estimates for first 10 samples:')
print('  (columns: [P(B), P(M)])')
for j in range(min(10, len(y_test))):
    print(f'  Sample {j+1:2d}: P(B)={proba[j,0]:.2f}, P(M)={proba[j,1]:.2f} -> {y_pred[j]}')

Probability estimates for first 10 samples:
  (columns: [P(B), P(M)])
  Sample  1: P(B)=0.90, P(M)=0.10 -> B
  Sample  2: P(B)=0.60, P(M)=0.40 -> B
  Sample  3: P(B)=0.60, P(M)=0.40 -> B
  Sample  4: P(B)=0.80, P(M)=0.20 -> B
  Sample  5: P(B)=0.80, P(M)=0.20 -> B
  Sample  6: P(B)=1.00, P(M)=0.00 -> B
  Sample  7: P(B)=0.00, P(M)=1.00 -> M
  Sample  8: P(B)=0.00, P(M)=1.00 -> M
  Sample  9: P(B)=0.00, P(M)=1.00 -> M
  Sample 10: P(B)=0.90, P(M)=0.10 -> B


## Conclusion

Just like in the fruit basket example, each decision tree independently classifies the data point. The class that receives the majority of votes from all trees is selected as the final prediction. In our results, 'B' (Benign) is predicted more often because it has more training samples, mirroring how the most frequently occurring fruit would be 'taken often' in the ensemble.